Local-only WGS compute: produce a 4x4 square trap-array phase payload
for the dedicated Windows hardware runner.

Parallel to ``scripts/sheet/testfile_sheet.py`` but uses WGS
(``slm.wgs.WGS_phase_generate``) on a rectangular 4x4 lattice target
(``SLM.target_generate("Rec", arraysize=[4, 4], translate=True)``)
instead of CGM on a continuous shape.  Same Fresnel + calibration
post-processing and same payload format as the sheet workflow, so the
unmodified Windows runner can display it.

Next step after running this script::

    ./push_run.sh payload/wgs_square/testfile_wgs_square_payload.npz


In [ ]:
from __future__ import annotations

import json
import os
import sys
import time

import numpy as np
import torch
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt

from slm.generation import SLM_class
from slm import imgpy
from slm import wgs


OUTPUT_DIR = "payload/wgs_square"
PAYLOAD_PATH = f"{OUTPUT_DIR}/testfile_wgs_square_payload.npz"
PARAMS_PATH = f"{OUTPUT_DIR}/testfile_wgs_square_params.json"
PREVIEW_PATH = f"{OUTPUT_DIR}/testfile_wgs_square_preview.pdf"


In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# --- Capture parameters (used later by the Windows runner) ---
etime_us = 4000        # 4 ms exposure
n_avg = 10             # average 10 frames
LUT = 207
fresnel_sd = 1000      # um -- best Fresnel shift distance from sweep


In [ ]:
# --- WGS parameters ---
array_nx, array_ny = 8, 8
wgs_loops = 100
wgs_threshold = 0.005


In [ ]:
# --- 1. SLM_class setup (4096^2 compute grid, matches legacy testfile.py) ---
SLM = SLM_class()
SLM.image_init(
    initGaussianPhase_user_defined=np.zeros((4096, 4096)),
    Plot=False,
)
W, H = SLM.SLMRes  # (1272, 1024)
cx, cy = W // 2, H // 2

correct = lambda screen: imgpy.SLM_screen_Correct(  # noqa: E731
    screen, LUT=LUT, correctionImgPath="calibration/CAL_LSH0905549_1013nm.bmp"
)

print(f"Compute grid: ({SLM.ImgResY}, {SLM.ImgResX})  "
      f"focal pitch = {SLM.Focalpitchx:.3f} um/px")
print(f"SLM native:   ({H}, {W})")
# 新增：目标相对零级光偏移（单位：focal plane pixel）
target_shift_fpx = 200
shift_um_x = target_shift_fpx * SLM.Focalpitchx
shift_um_y = target_shift_fpx * SLM.Focalpitchy


In [ ]:
# --- 2. Generate 4x4 rectangular trap-array target ---
targetAmp = SLM.target_generate(
    "Rec",
    arraysize=[array_nx, array_ny],
    distance=[-shift_um_x, -shift_um_y],
    translate=False,
    Plot=False,
)
print(
    f"\n[TARGET] 4x4 square lattice  dtype={targetAmp.dtype} "
    f"shift={shift_um_x:.3f} um, {shift_um_y:.3f} um"
    f"shape={targetAmp.shape} nonzero={np.count_nonzero(targetAmp)}"
)


In [ ]:
# Inline target visualization (amplitude + phase, full + auto-zoom).
amp = np.abs(targetAmp)
phase_t = np.angle(targetAmp)
ny, nx = amp.shape
x_um = (np.arange(nx) - (nx - 1) / 2.0) * SLM.Focalpitchx
y_um = (np.arange(ny) - (ny - 1) / 2.0) * SLM.Focalpitchy
extent_full = [x_um[0], x_um[-1], y_um[-1], y_um[0]]

peak = float(amp.max())
support = amp >= (peak * 0.05) if peak > 0 else np.zeros_like(amp, dtype=bool)
if not np.any(support):
    support = amp > 0
rows, cols = np.where(support)
if rows.size > 0:
    margin = 10
    r0 = max(0, rows.min() - margin)
    r1 = min(ny, rows.max() + margin + 1)
    c0 = max(0, cols.min() - margin)
    c1 = min(nx, cols.max() + margin + 1)
else:
    r0, r1, c0, c1 = 0, ny, 0, nx
extent_zoom = [x_um[c0], x_um[c1 - 1], y_um[r1 - 1], y_um[r0]]

plt.figure(figsize=(10, 8))
plt.subplot(2, 2, 1)
plt.imshow(amp, cmap="magma", extent=extent_full, aspect="auto")
plt.colorbar(label="|targetAmp|"); plt.title("target amplitude")
plt.xlabel("x (um)"); plt.ylabel("y (um)")
plt.subplot(2, 2, 2)
plt.imshow(phase_t, cmap="twilight", extent=extent_full, aspect="auto")
plt.colorbar(label="phase (rad)"); plt.title("target phase")
plt.xlabel("x (um)"); plt.ylabel("y (um)")
plt.subplot(2, 2, 3)
plt.imshow(amp[r0:r1, c0:c1], cmap="magma", extent=extent_zoom, aspect="auto")
plt.colorbar(label="|targetAmp|"); plt.title("amplitude (zoom)")
plt.xlabel("x (um)"); plt.ylabel("y (um)")
plt.subplot(2, 2, 4)
plt.imshow(phase_t[r0:r1, c0:c1], cmap="twilight", extent=extent_zoom, aspect="auto")
plt.colorbar(label="phase (rad)"); plt.title("phase (zoom)")
plt.xlabel("x (um)"); plt.ylabel("y (um)")
plt.tight_layout(); plt.show()


In [ ]:
# --- 3. Run WGS (zero-phase init, deterministic) ---
init_phase = torch.zeros((4096, 4096), dtype=torch.float32)
print(f"\n[WGS] running {wgs_loops} loops on 4096x4096 grid...")
t0 = time.perf_counter()
SLM_Phase = wgs.WGS_phase_generate(
    torch.tensor(SLM.initGaussianAmp),
    init_phase,
    torch.from_numpy(targetAmp),
    Loop=wgs_loops,
    threshold=wgs_threshold,
    Plot=False,
)
wgs_wall_time = time.perf_counter() - t0
per_iter_ms = wgs_wall_time / max(wgs_loops, 1) * 1000
print(f"[WGS] done in {wgs_wall_time:.2f} s ({per_iter_ms:.1f} ms/iter)")

phase_np = SLM_Phase.cpu().clone().numpy()
SLM_screen_raw = SLM.phase_to_screen(phase_np)


In [ ]:
# --- 4. Post-hoc Fresnel lens ---
fresnel = SLM.fresnel_lens_phase_generate(fresnel_sd, cx, cy)[0]
SLM_screen_shift = (
    (SLM_screen_raw.astype(np.int32) + fresnel.astype(np.int32)) % 256
).astype(np.uint8)


In [ ]:
# --- 5. Calibration correction ---
SLM_screen_final = correct(SLM_screen_shift)
print(f"\n[SCREEN] ready-to-display uint8 {SLM_screen_final.shape} "
      f"range=[{SLM_screen_final.min()}, {SLM_screen_final.max()}]")


In [ ]:
# Inline view of the final uint8 SLM screen.
plt.figure(figsize=(8, 6))
plt.imshow(SLM_screen_final, cmap="gray", vmin=0, vmax=255)
plt.colorbar(label="uint8")
plt.title(f"SLM screen (Fresnel + calibration applied) {SLM_screen_final.shape}")
plt.tight_layout(); plt.show()


In [ ]:
# --- 6. Diagnostic metrics ---
from slm.metrics import fidelity as _fidelity, efficiency as _efficiency
from slm.propagation import fft_propagate
from slm.targets import measure_region as _measure_region

region_np = _measure_region(targetAmp.shape, targetAmp, margin=5)
E_out = fft_propagate(SLM.initGaussianAmp * np.exp(1j * phase_np))
F = _fidelity(E_out, targetAmp, region_np)
eta = _efficiency(E_out, region_np)
print(f"[METRICS] fidelity F={F:.6f}  efficiency eta={eta*100:.2f}%")


In [ ]:
# --- 7. Save payload.npz ---
np.savez_compressed(PAYLOAD_PATH, slm_screen=SLM_screen_final)
payload_size_kb = int(np.ceil(os.path.getsize(PAYLOAD_PATH) / 1024))
print(f"\n[SAVE] payload:  {PAYLOAD_PATH}  ({payload_size_kb} KB)")


In [ ]:
# --- 8. Save params.json ---
params = {
    "algorithm": "WGS",
    "payload": PAYLOAD_PATH,
    "compute_grid": [int(SLM.ImgResY), int(SLM.ImgResX)],
    "slm_native": [int(H), int(W)],
    "focal_pitch_x_um_per_px": round(float(SLM.Focalpitchx), 4),
    "focal_pitch_y_um_per_px": round(float(SLM.Focalpitchy), 4),
    "runner_defaults": {
        "etime_us": etime_us,
        "n_avg": n_avg,
        "monitor": 1,
    },
    "fresnel_applied_on_linux": True,
    "fresnel_shift_distance_um": fresnel_sd,
    "fresnel_center_xy_px": [int(cx), int(cy)],
    "calibration_applied_on_linux": True,
    "calibration_bmp": "calibration/CAL_LSH0905549_1013nm.bmp",
    "LUT": LUT,
    "target": "rectangular_lattice",
    "array_nx": array_nx,
    "array_ny": array_ny,
    "wgs_loops": wgs_loops,
    "wgs_threshold": wgs_threshold,
    "wgs_wall_time_s": round(wgs_wall_time, 3),
    "wgs_per_iter_ms": round(per_iter_ms, 2),
    "final_fidelity": round(float(F), 6),
    "final_efficiency": round(float(eta), 6),
    "timestamp_local": time.strftime("%Y-%m-%dT%H:%M:%S", time.localtime()),
}
with open(PARAMS_PATH, "w", encoding="utf-8") as f:
    json.dump(params, f, ensure_ascii=False, indent=2)
print(f"[SAVE] params:  {PARAMS_PATH}")


In [ ]:
# --- 9. Save preview.pdf ---
# 6-panel inline diagnostic (replaces the legacy preview PDF).
from slm.targets import mask_from_target
target_mask = mask_from_target(targetAmp)
try:
    E_out_diag = E_out_4096
except NameError:
    E_out_diag = E_out_1024
try:
    slm_phase_full_diag = SLM_Phase.cpu().numpy()
except (NameError, AttributeError):
    slm_phase_full_diag = phase_wrapped

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes[0, 0].imshow(SLM.initGaussianAmp, cmap="viridis")
axes[0, 0].set_title("Input amplitude |S|")
axes[0, 1].imshow(np.abs(targetAmp) ** 2, cmap="hot")
axes[0, 1].set_title("Target |tau|^2 (WGS square)")
target_phase_masked = np.where(target_mask > 0, np.angle(targetAmp), np.nan)
axes[0, 2].imshow(target_phase_masked, cmap="twilight", vmin=-np.pi, vmax=np.pi)
axes[0, 2].set_title("Target phase arg(tau)")
out_int = (np.abs(E_out_diag) ** 2) * region_np
axes[1, 0].imshow(out_int, cmap="hot")
axes[1, 0].set_title(f"Output |E_out|^2\nF={F:.4f}, eta={100*eta:.2f}%")
out_phase_masked = np.where(target_mask > 0, np.angle(E_out_diag), np.nan)
axes[1, 1].imshow(out_phase_masked, cmap="twilight", vmin=-np.pi, vmax=np.pi)
axes[1, 1].set_title("Output phase arg(E_out)")
axes[1, 2].imshow(SLM_screen_final, cmap="gray", vmin=0, vmax=255)
axes[1, 2].set_title(f"SLM screen (Fresnel+calib applied)\n{SLM_screen_final.shape} uint8")
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()


In [ ]:
# --- 10. Next-step hint ---
print()
print("=" * 72)
print("Payload ready.  Next step (pushes to Windows and runs the experiment):")
print()
print(f"    ./push_run.sh {PAYLOAD_PATH}")
print()
print("=" * 72)
